# JUMP CRISPR arm

{func}`~mantispy.ds.jump_crispr` serves the CRISPR arm of `cpg0016-jump-assembled` {cite:p}`Chandrasekaran_2023`, 51,185 wells of knockouts in U2OS cells, corrected for well position and cell count and feature-selected by jump-profiling-recipe. Each gene is knocked out by several guides, and the guides are its replicates, so this is a well-level CRISPR screen, the counterpart to the single-cell optical pooled screens {func}`~mantispy.ds.scallops_arv471` and {func}`~mantispy.ds.cp_posh`.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import scanpy as sc

import mantispy as mt

In [2]:
import plotly.io as pio

pio.renderers.default = "notebook_connected"

The well-level object is large, so this page uses the gene-level consensus, one `modz` profile per gene over its guides.

In [3]:
adata = mt.ds.jump_crispr(aggregated=True)
adata

AnnData object with n_obs × n_vars = 7974 × 595
    obs: 'Metadata_Gene', 'Metadata_ReplicateCount', 'Metadata_Source', 'Metadata_JCP2022', 'Metadata_Control_Type', 'Metadata_Perturbation', 'Metadata_Perturbation_Type', 'Metadata_Control', 'Metadata_ChromosomeArm'
    var: 'object', 'feature_group', 'feature', 'channel', 'scale', 'angle', 'gray_levels', 'radial_bin', 'params', 'is_feature'
    uns: 'mantispy'
    layers: None (.X)

## Controls against treatments

`Metadata_Control_Type` labels each gene consensus a negative control (`negcon`), a positive control (`poscon`) or a treatment (`trt`). On the first two principal components the controls should sit apart from the bulk of the genes.

In [4]:
scaled = adata.copy()
sc.pp.scale(scaled)
scaled.X = np.nan_to_num(scaled.X)
sc.pp.pca(scaled, n_comps=10)
coords = pd.DataFrame(scaled.obsm["X_pca"][:, :2], columns=["PC1", "PC2"])
coords["gene"] = adata.obs["Metadata_Gene"].to_numpy()
coords["control type"] = adata.obs["Metadata_Control_Type"].astype(str).to_numpy()
fig = px.scatter(
    coords,
    x="PC1",
    y="PC2",
    color="control type",
    hover_name="gene",
    title="JUMP CRISPR genes in profile space",
    opacity=0.5,
)
fig.show()